<a href="https://colab.research.google.com/github/Feellived/molecular-reliability-signals/blob/yoonsoo/01_verify_team_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · 팀 프로필 검증 (태스크 2·4·5·6 재현)

**연구계획서 대응**: 5.4절(변형 가능성 사전 선별)

---

## 왜 이 노트북이 필요한가

팀원이 `molecule_profile.csv` 로 **부모 분자(2) · 골격(4) · 변형 가능성(5) · 게이트(6)** 를 이미 끝냈다.
그대로 쓰면 되지만, **10% 게이트 판정은 "어떤 변형 축을 논문에서 쓸지"를 결정하는 근거**다.
근거가 되는 수치는 한 번은 독립적으로 재현해 두는 편이 안전하다.

그래서 이 노트북은 **다시 계산하는 게 목적이 아니라 대조가 목적**이다.
같은 분자에 대해 우리가 직접 계산한 값과 팀원 값이 얼마나 일치하는지 본다.

| 항목 | 팀원 열 | 우리가 재계산하는 방법 |
|---|---|---|
| 부모 분자 | `parent_smiles` | RDKit `Cleanup` + `LargestFragmentChooser` |
| 골격 | `scaffold` | `MurckoScaffold.GetScaffoldForMol` |
| 호변이성질체 수 | `n_tautomers` | RDKit `TautomerEnumerator` (규칙 기반, "가능한 걸 다 보여줌") |
| 양성자화 상태 수 | `n_protomers` | `dimorphite-dl` (pH 창 안에서 "그럴듯한 것만 추림") |
| 염 형태 | `has_salt` | 원본 SMILES 에 마침표(`.`)가 있는가 |
| 입체 표기 | `has_stereo` | 원본 SMILES 에 `@`, `/`, `\` 가 있는가 |

> **개수가 정확히 일치하지 않는 것은 정상이다.**
> 호변이성질체 열거는 `maxTautomers` 상한값에 따라, 양성자화는 pH 범위와 pKa 정밀도 설정에 따라
> 결과 개수가 달라진다. 팀원의 설정값을 모르므로 **"2개 이상인가"라는 판정(=게이트에 실제로 쓰이는 값)의
> 일치율**을 1차 지표로 본다. 개수 자체의 일치율은 참고로만 본다.

## 두 가지 모드

- `MODE = "verify"` (기본): 물성당 300개 표본만 대조. 5~8분.
- `MODE = "full"`: 전수 재계산. 30~40분. **게이트 수치를 우리 손으로 다시 산출해야 할 때**
  (예: 허용성 표에서 `hia_hou` 의 pH 창을 넓히기로 결정한 경우) 이 모드를 쓴다.

In [ ]:
# [셀 1] 패키지 설치 (Colab 세션이 끊기면 다시 실행)
%pip install -q rdkit dimorphite-dl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.5 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# [셀 2] 드라이브 마운트 + 경로  ← 모든 노트북 공통
# ----------------------------------------------------------------------------
# 읽는 곳과 쓰는 곳이 다르다.
#   data/processed 는 팀원이 공유한 폴더라 쓰기가 막혀 있다(Read-only file system).
#   팀 파일은 읽기만 하고, 우리 결과는 MIST/outputs 아래에 쓴다.
# 로컬에서 돌린다면 drive.mount 두 줄을 지우고 DATA_ROOT / OUT_ROOT 만 바꾸면 된다.
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/MIST/data")
RAW_DIR   = DATA_ROOT / "raw"             # [읽기] TDC 원본
TEAM_DIR  = DATA_ROOT / "processed"       # [읽기] 팀원 산출물

OUT_ROOT   = Path("/content/drive/MyDrive/MIST/outputs")
PROC_DIR   = OUT_ROOT / "processed"       # [쓰기] 물성별 우리 산출물
REPORT_DIR = PROC_DIR / "reports"         # [쓰기] 집계표
CACHE_DIR  = PROC_DIR / "_cache"          # [쓰기] 무거운 계산 캐시
for _d in (PROC_DIR, REPORT_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

SEED = 42     # 분할·샘플링 재현용

print("[읽기] RAW  :", RAW_DIR, "(있음)" if RAW_DIR.exists() else "(없음 — 경로 확인)")
print("[읽기] TEAM :", TEAM_DIR, "(있음)" if TEAM_DIR.exists() else "(없음 — 경로 확인)")
print("[쓰기] OUT  :", PROC_DIR)

Mounted at /content/drive
[읽기] RAW  : /content/drive/MyDrive/MIST/data/raw (있음)
[읽기] TEAM : /content/drive/MyDrive/MIST/data/processed (있음)
[쓰기] OUT  : /content/drive/MyDrive/MIST/outputs/processed


In [ ]:
# ============================================================================
# [셀 3] TDC ADMET Benchmark Group 22종 메타데이터 (참조용 상수)
# ----------------------------------------------------------------------------
# 연구계획서 5.1절: 회귀 9종 + 분류 13종 = 22종.
# 아래 dict 는 "우리가 알고 있는 정답"이며, 실제 데이터에서 자동 판별한 결과와
# 대조해서 틀리면 경고를 띄우는 용도로 쓴다(하드코딩만 믿지 않는다).
# ============================================================================
TASK_TYPE_REF = {
    # ---- 회귀 9종 (Y 가 연속값) ----
    "caco2_wang":                 "regression",   # Caco-2 세포 투과도 log(cm/s)
    "lipophilicity_astrazeneca":  "regression",   # logD (pH 7.4 옥탄올/물 분배계수)
    "solubility_aqsoldb":         "regression",   # logS 수용해도
    "ppbr_az":                    "regression",   # 혈장단백결합률 (%)
    "vdss_lombardo":              "regression",   # 정상상태 분포용적 (L/kg)
    "half_life_obach":            "regression",   # 반감기 (h)
    "clearance_hepatocyte_az":    "regression",   # 간세포 청소율 (uL/min/1e6 cells)
    "clearance_microsome_az":     "regression",   # 마이크로솜 청소율 (mL/min/g)
    "ld50_zhu":                   "regression",   # 급성독성 LD50 (-log mol/kg)
    # ---- 분류 13종 (Y 가 0/1) ----
    "hia_hou":                            "classification",  # 장 흡수 여부
    "pgp_broccatelli":                    "classification",  # P-gp 저해 여부
    "bioavailability_ma":                 "classification",  # 경구 생체이용률
    "bbb_martins":                        "classification",  # 혈뇌장벽 투과
    "cyp2c9_veith":                       "classification",  # CYP2C9 저해
    "cyp2d6_veith":                       "classification",  # CYP2D6 저해
    "cyp3a4_veith":                       "classification",  # CYP3A4 저해
    "cyp2c9_substrate_carbonmangels":     "classification",  # CYP2C9 기질
    "cyp2d6_substrate_carbonmangels":     "classification",  # CYP2D6 기질
    "cyp3a4_substrate_carbonmangels":     "classification",  # CYP3A4 기질
    "herg":                               "classification",  # hERG 심독성
    "ames":                               "classification",  # Ames 변이원성
    "dili":                               "classification",  # 약물유발 간손상
}
assert len(TASK_TYPE_REF) == 22
print("회귀:", sum(v == "regression" for v in TASK_TYPE_REF.values()),
      "/ 분류:", sum(v == "classification" for v in TASK_TYPE_REF.values()))

회귀: 9 / 분류: 13


## 1. 재계산 함수들

각 함수 끝에 동작 확인 예시가 붙어 있다. 결과를 눈으로 보고 넘어가면 된다.

**부모 분자 뽑는 순서**

| 단계 | RDKit | 하는 일 |
|---|---|---|
| 파싱 | `Chem.MolFromSmiles` | 문자열 → 분자. **실패하면 `None`** = 무효 분자 |
| 정리 | `rdMolStandardize.Cleanup` | 금속 결합 끊기, 표기 정규화, 원자가 재조정, 입체 재할당 |
| 부모 선택 | `LargestFragmentChooser` | 가장 큰 조각만 남긴다 = **탈염** |
| 정준화 | `Chem.MolToSmiles` | 같은 분자면 항상 같은 문자열 |

`Cleanup` 은 이온화 상태를 살짝 손볼 수 있다(reionize). 양성자화 상태 자체는 변형 축(B1)이므로
여기서 중성화(uncharge)는 **하지 않는다**.

In [ ]:
# ============================================================================
# [셀 4] RDKit 준비 + 부모 분자 추출 함수
# ============================================================================
import pandas as pd, numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

# RDKit 은 이상한 분자를 만나면 경고를 쏟아낸다. 우리는 결과를 직접 집계하므로 로그는 끈다.
RDLogger.DisableLog("rdApp.*")

# 가장 큰 조각을 고르는 도구.
#   preferOrganic=True : 크기가 비슷하면 유기 조각을 우선한다 (RDKit 버전에 따라 인자가 없을 수 있어 예외 처리)
try:
    _LFC = rdMolStandardize.LargestFragmentChooser(preferOrganic=True)
except Exception:
    _LFC = rdMolStandardize.LargestFragmentChooser()


def to_parent(smiles: str) -> dict:
    """SMILES 문자열 하나 -> 부모 분자 정보 dict.

    실패해도 예외를 던지지 않고 note 에 이유를 적어 돌려준다.
    (22종 8만 건을 돌리는 중에 한 건 때문에 멈추면 곤란하므로)
    """
    out = {
        "parse_ok": False,        # RDKit 이 읽었는가
        "n_fragments": np.nan,    # 연결되지 않은 조각 수 (2 이상이면 염/혼합물)
        "smiles_parent": None,    # 가장 큰 조각의 정준 SMILES
        "inchikey_parent": None,  # 교차 확인용 표준 식별자
        "n_heavy_atoms": np.nan,  # 부모 분자의 무거운 원자(수소 제외) 수
        "note": "",
    }
    if not isinstance(smiles, str) or smiles.strip() == "":
        out["note"] = "empty_string"
        return out

    # ---- 1) 파싱 ----
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out["note"] = "parse_failed"     # 여기 걸리면 무효 분자로 확정
        return out
    if mol.GetNumAtoms() == 0:
        out["note"] = "no_atoms"
        return out

    # ---- 2) 표준 정리 ----
    try:
        clean = rdMolStandardize.Cleanup(mol)
    except Exception as e:
        clean = mol
        out["note"] = f"cleanup_failed:{type(e).__name__}"

    # ---- 3) 조각 수 세기 ----
    try:
        frags = Chem.GetMolFrags(clean, asMols=True, sanitizeFrags=True)
    except Exception:
        frags = (clean,)
    out["n_fragments"] = len(frags)

    # ---- 4) 가장 큰 조각 = 부모 분자 ----
    try:
        parent = _LFC.choose(clean) if len(frags) > 1 else clean
    except Exception:
        # 예비책: 무거운 원자 수가 가장 많은 조각을 직접 고른다
        parent = max(frags, key=lambda m: m.GetNumHeavyAtoms())

    if parent is None or parent.GetNumAtoms() == 0:
        out["note"] = "parent_empty"
        return out

    try:
        Chem.SanitizeMol(parent)
    except Exception as e:
        out["note"] = f"sanitize_failed:{type(e).__name__}"
        return out

    # ---- 5) 정준 SMILES (isomericSmiles=True 가 기본 → 입체 정보 유지) ----
    out["smiles_parent"] = Chem.MolToSmiles(parent)
    out["n_heavy_atoms"] = parent.GetNumHeavyAtoms()
    out["parse_ok"] = True
    try:
        out["inchikey_parent"] = Chem.MolToInchiKey(parent)
    except Exception:
        pass
    return out


# ---- 함수가 제대로 도는지 눈으로 확인 ----
demo = [
    ("아스피린 (조각 1개)",        "CC(=O)Oc1ccccc1C(=O)O"),
    ("니코틴 염산염 (조각 2개)",   "C[C@H]1CCCN1c1cccnc1.Cl"),
    ("나트륨염",                   "CC(=O)[O-].[Na+]"),
    ("깨진 SMILES",                "C1CC"),
]
for label, smi in demo:
    r = to_parent(smi)
    print(f"{label:28s} parse_ok={str(r['parse_ok']):5s} "
          f"frags={r['n_fragments']} parent={r['smiles_parent']} {r['note']}")

아스피린 (조각 1개)                 parse_ok=True  frags=1 parent=CC(=O)Oc1ccccc1C(=O)O 
니코틴 염산염 (조각 2개)              parse_ok=True  frags=2 parent=C[C@H]1CCCN1c1cccnc1 
나트륨염                         parse_ok=True  frags=2 parent=CC(=O)[O-] 
깨진 SMILES                    parse_ok=False frags=nan parent=None parse_failed


In [ ]:
import pandas as pd
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog("rdApp.*")

ROOT = Path("/content/drive/MyDrive/MIST/data")
un = rdMolStandardize.Uncharger()

def neutral(s):
    m = Chem.MolFromSmiles(s)
    return Chem.MolToSmiles(un.uncharge(m)) if m else s

rows = []
for prof_path in sorted((ROOT / "processed").glob("*/molecule_profile.csv")):
    d = prof_path.parent.name
    parents = pd.read_csv(prof_path).parent_smiles.dropna()
    uniq = set(parents)
    charged = {p for p in uniq if ("+" in p or "-" in p)}
    # 전하만 다른 짝이 실제로 존재하는가: 중성화했더니 다른 부모와 같아지는 경우
    collide = {p for p in charged if neutral(p) != p and neutral(p) in uniq}
    rows.append({"dataset": d, "고유부모": len(uniq), "전하보유": len(charged),
                 "중성형이_이미존재": len(collide)})

chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print(f"\n전체 추가 중복 후보: {chk.중성형이_이미존재.sum()}건")

                       dataset  고유부모  전하보유  중성형이_이미존재
                          ames  7255  1800          0
                   bbb_martins  1966   187          0
            bioavailability_ma   640    86          0
                    caco2_wang   905   145          1
       clearance_hepatocyte_az  1020   356          0
        clearance_microsome_az  1102   354          0
cyp2c9_substrate_carbonmangels   666    87          0
                  cyp2c9_veith 12053  3696          4
cyp2d6_substrate_carbonmangels   664    87          0
                  cyp2d6_veith 13093  3995          5
cyp3a4_substrate_carbonmangels   667    87          0
                  cyp3a4_veith 12300  3801          5
                          dili   475    73          1
               half_life_obach   665   107          0
                          herg   642   345         24
                       hia_hou   578    77          0
                      ld50_zhu  7342   726          0
     lipophilicity_astrazene

In [ ]:
# ============================================================================
# [셀 5] Bemis-Murcko 골격
# ----------------------------------------------------------------------------
# 분자에서 곁가지를 다 떼고 고리 중심 뼈대만 남긴 것.
#   아스피린   CC(=O)Oc1ccccc1C(=O)O        -> c1ccccc1
#   이부프로펜 CC(C)Cc1ccc(cc1)C(C)C(=O)O   -> c1ccccc1
# 전혀 다른 약이지만 골격은 같다. 노트북 02의 분할은 이 단위로 묶어서 쪼갠다.
# includeChirality=False : 골격 분할은 관례적으로 입체를 무시한다.
# ============================================================================
from rdkit.Chem.Scaffolds import MurckoScaffold

def get_scaffold(smiles_parent):
    mol = Chem.MolFromSmiles(smiles_parent) if isinstance(smiles_parent, str) else None
    if mol is None:
        return None
    try:
        return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol), isomericSmiles=False)
    except Exception:
        return None

for label, smi in [("아스피린", "CC(=O)Oc1ccccc1C(=O)O"),
                   ("이부프로펜", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
                   ("카페인", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
                   ("에탄올(고리없음)", "CCO")]:
    print(f"{label:18s} {smi:35s} -> 골격: {get_scaffold(smi)!r}")

아스피린               CC(=O)Oc1ccccc1C(=O)O               -> 골격: 'c1ccccc1'
이부프로펜              CC(C)Cc1ccc(cc1)C(C)C(=O)O          -> 골격: 'c1ccccc1'
카페인                Cn1cnc2c1c(=O)n(C)c(=O)n2C          -> 골격: 'O=c1[nH]c(=O)c2[nH]cnc2[nH]1'
에탄올(고리없음)          CCO                                 -> 골격: ''


In [ ]:
# ============================================================================
# [셀 6] 호변이성질체 열거기 준비 + 동작 확인
# ============================================================================
import pandas as pd, numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog("rdApp.*")

MAX_TAUTOMERS = 32        # 분자당 열거 상한 (조합 폭발 방지)

# 열거기 객체는 '지연 생성'한다. 병렬 실행 시 이 객체가 자식 프로세스로 복사되면
# 직렬화(pickle)에 실패하므로, 각 프로세스가 자기 것을 따로 만들게 하기 위해서다.
_TAUT_ENUM = None

def _get_taut_enum():
    global _TAUT_ENUM
    if _TAUT_ENUM is None:
        p = rdMolStandardize.CleanupParameters()
        p.maxTautomers = MAX_TAUTOMERS
        _TAUT_ENUM = rdMolStandardize.TautomerEnumerator(p)
    return _TAUT_ENUM


def count_tautomers(smiles):
    """부모 분자 SMILES -> (호변이성질체 개수, 상한도달여부)."""
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    if mol is None:
        return (np.nan, False)
    try:
        res = _get_taut_enum().Enumerate(mol)
        smis = list(res.smiles)          # 열거된 형태들의 SMILES 튜플
        # status 가 0(Completed) 이 아니면 상한/시간에 걸려 중단된 것
        hit_cap = int(getattr(res, "status", 0)) != 0 or len(smis) >= MAX_TAUTOMERS
        return (len(set(smis)), hit_cap)
    except Exception:
        return (np.nan, False)


# ---- 동작 확인: 아세틸아세톤은 케토형/엔올형이 있는 교과서 예시 ----
for label, smi in [("아세틸아세톤(케토-엔올)", "CC(=O)CC(=O)C"),
                   ("2-피리돈(락탐-락팀)",      "O=c1cccc[nH]1"),
                   ("벤젠(변형 불가)",          "c1ccccc1")]:
    n, cap = count_tautomers(smi)
    print(f"{label:24s} 호변이성질체 {n}개 (상한도달={cap})")

print("\n아세틸아세톤이 실제로 만들어내는 형태들:")
for s in sorted(set(_get_taut_enum().Enumerate(Chem.MolFromSmiles("CC(=O)CC(=O)C")).smiles)):
    print("   ", s)

아세틸아세톤(케토-엔올)            호변이성질체 5개 (상한도달=False)
2-피리돈(락탐-락팀)             호변이성질체 3개 (상한도달=False)
벤젠(변형 불가)                호변이성질체 1개 (상한도달=False)

아세틸아세톤이 실제로 만들어내는 형태들:
    C=C(O)C=C(C)O
    C=C(O)CC(=C)O
    C=C(O)CC(C)=O
    CC(=O)C=C(C)O
    CC(=O)CC(C)=O


In [ ]:
# ============================================================================
# [셀 7] dimorphite-dl 어댑터 (버전 차이 흡수 + 실패 시 대체 경로)
# ============================================================================
PH_MIN, PH_MAX = 6.4, 8.4     # 생리적 pH 7.4 +- 1
PKA_PRECISION  = 1.0          # pKa 추정 불확실성 (클수록 더 많은 상태를 만든다)
MAX_PROTOMERS  = 16

_PROTONATOR = None            # 프로세스마다 지연 초기화되는 실제 호출 함수
_PROTO_METHOD = None          # 어떤 경로를 썼는지 기록용 문자열


def _init_protonator():
    """설치된 dimorphite-dl 버전에 맞는 호출 함수를 만든다."""
    global _PROTONATOR, _PROTO_METHOD

    # (A) 2.x 스타일: 모듈 수준 함수
    try:
        from dimorphite_dl import protonate_smiles as _ps
        def _call(smi):
            try:
                return list(_ps(smi, ph_min=PH_MIN, ph_max=PH_MAX,
                                precision=PKA_PRECISION, max_variants=MAX_PROTOMERS))
            except TypeError:
                return list(_ps(smi))
        _PROTONATOR, _PROTO_METHOD = _call, "dimorphite_dl.protonate_smiles"
        return
    except Exception:
        pass

    # (B) 1.x / 2.x 클래스 스타일
    try:
        from dimorphite_dl import DimorphiteDL
        try:
            _dl = DimorphiteDL(min_ph=PH_MIN, max_ph=PH_MAX,
                               max_variants=MAX_PROTOMERS,
                               pka_precision=PKA_PRECISION, label_states=False, silent=True)
        except TypeError:
            _dl = DimorphiteDL(min_ph=PH_MIN, max_ph=PH_MAX, pka_precision=PKA_PRECISION)
        _PROTONATOR, _PROTO_METHOD = (lambda smi: list(_dl.protonate(smi))), "DimorphiteDL.protonate"
        return
    except Exception:
        pass

    # (C) 최후의 대체: 이온화 가능 작용기가 있으면 '상태 2개 이상'으로 간주하는 간이 판정
    #     정확한 열거가 아니므로 결과 열에 method 를 반드시 남긴다.
    IONIZABLE = [Chem.MolFromSmarts(p) for p in [
        "[CX3](=O)[OX2H1]",        # 카복실산
        "[SX4](=O)(=O)[NX3H1,NX3H2]",  # 설폰아마이드
        "c1nnn[nH]1",              # 테트라졸
        "[OX2H1][cX3]",            # 페놀
        "[NX3;H2,H1;!$(NC=O);!$(NS=O);!$(N[a])]",  # 1,2급 아민
        "[NX3;H0;!$(NC=O);!$(NS=O);!$(N[a]);!$([N+])]",  # 3급 아민
    ]]
    def _fallback(smi):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return [smi]
        hit = any(mol.HasSubstructMatch(p) for p in IONIZABLE if p is not None)
        return [smi, smi] if hit else [smi]   # 개수만 의미 있는 대체값
    _PROTONATOR, _PROTO_METHOD = _fallback, "FALLBACK_smarts_only"


def count_protomers(smiles):
    """부모 분자 SMILES -> (지정 pH 범위에서 가능한 양성자화 상태 개수)."""
    global _PROTONATOR
    if _PROTONATOR is None:
        _init_protonator()
    if not isinstance(smiles, str):
        return np.nan
    try:
        out = _PROTONATOR(smiles)
    except Exception:
        return np.nan
    # dimorphite 결과를 RDKit 정준 SMILES 로 바꿔 중복을 제거한다
    canon = set()
    for s in out:
        m = Chem.MolFromSmiles(s)
        if m is not None:
            canon.add(Chem.MolToSmiles(m))
    return len(canon) if canon else np.nan


_init_protonator()
print("사용 중인 양성자화 계산 경로:", _PROTO_METHOD)
if _PROTO_METHOD.startswith("FALLBACK"):
    print("  [경고] dimorphite-dl 을 불러오지 못했습니다. 개수는 근사치이며 B1-양성자화 결과 해석에 주의.")

for label, smi in [("아세트산(산성)",     "CC(=O)O"),
                   ("펜에틸아민(염기성)", "NCCc1ccccc1"),
                   ("벤젠(이온화 없음)",  "c1ccccc1")]:
    print(f"{label:20s} 양성자화 상태 {count_protomers(smi)}개  (pH {PH_MIN}~{PH_MAX})")

사용 중인 양성자화 계산 경로: dimorphite_dl.protonate_smiles
아세트산(산성)             양성자화 상태 1개  (pH 6.4~8.4)
펜에틸아민(염기성)           양성자화 상태 2개  (pH 6.4~8.4)
벤젠(이온화 없음)           양성자화 상태 1개  (pH 6.4~8.4)


In [ ]:
for s in ["CC(=O)O", "NCCc1ccccc1", "c1ccccc1"]:
    print(f"{s:16s} -> {_PROTONATOR(s)}")

CC(=O)O          -> ['CC(=O)[O-]']
NCCc1ccccc1      -> ['[NH3+]CCc1ccccc1', 'NCCc1ccccc1']
c1ccccc1         -> ['c1ccccc1']


In [ ]:
# ============================================================================
# [셀 8] 염 / 입체 표기 판정 함수
# ============================================================================
from rdkit.Chem import rdMolDescriptors

STEREO_CHARS = ("@", "/", "\\")     # 역슬래시는 파이썬 문자열에서 두 번 써야 한 글자다

def has_salt_raw(smiles_original):
    """원본 SMILES 에 마침표가 있으면 염/혼합물로 등록된 것."""
    return isinstance(smiles_original, str) and ("." in smiles_original)

def has_stereo_raw(smiles_original):
    """원본 SMILES 에 입체 기호가 하나라도 있으면 입체 표기 보유."""
    return isinstance(smiles_original, str) and any(c in smiles_original for c in STEREO_CHARS)

def stereo_center_counts(smiles_parent):
    """(전체 원자 입체중심 수, 미지정 입체중심 수). 실패하면 (nan, nan)."""
    mol = Chem.MolFromSmiles(smiles_parent) if isinstance(smiles_parent, str) else None
    if mol is None:
        return (np.nan, np.nan)
    try:
        n_all = rdMolDescriptors.CalcNumAtomStereoCenters(mol)
        n_unspec = rdMolDescriptors.CalcNumUnspecifiedAtomStereoCenters(mol)
        return (n_all, n_unspec)
    except Exception:
        # 위 함수는 입체 정보가 미리 할당돼 있어야 동작한다. 실패하면 직접 할당 후 재시도.
        try:
            Chem.AssignStereochemistry(mol, cleanIt=True, force=True, flagPossibleStereoCenters=True)
            centers = Chem.FindMolChiralCenters(mol, includeUnassigned=True, useLegacyImplementation=False)
            return (len(centers), sum(1 for _, c in centers if c == "?"))
        except Exception:
            return (np.nan, np.nan)


for label, smi in [("L-알라닌",        "C[C@@H](N)C(=O)O"),
                   ("알라닌(입체없음)", "CC(N)C(=O)O"),
                   ("니코틴 염산염",    "C[C@H]1CCCN1c1cccnc1.Cl"),
                   ("트랜스-2-부텐",    "C/C=C/C")]:
    print(f"{label:16s} 염={str(has_salt_raw(smi)):5s} 입체표기={str(has_stereo_raw(smi)):5s} "
          f"입체중심(전체,미지정)={stereo_center_counts(smi)}")

L-알라닌            염=False 입체표기=True  입체중심(전체,미지정)=(1, 0)
알라닌(입체없음)        염=False 입체표기=False 입체중심(전체,미지정)=(1, 1)
니코틴 염산염          염=True  입체표기=True  입체중심(전체,미지정)=(1, 0)
트랜스-2-부텐         염=False 입체표기=True  입체중심(전체,미지정)=(0, 0)


## 2. 표본 뽑아 재계산

계산 결과는 `_cache/verify_cache.csv` 에 저장되므로, 런타임이 끊겨도 이어서 돌릴 수 있다.

In [ ]:
# ============================================================================
# [셀 9] 대조할 분자 뽑기 + 재계산
# ============================================================================
from tqdm.auto import tqdm

MODE            = "verify"   # "verify"(표본 대조) 또는 "full"(전수 재계산)
SAMPLE_PER_SET  = 300        # verify 모드에서 물성당 표본 크기
N_JOBS          = 2          # 병렬 프로세스 수. 문제가 생기면 1로.

DATASETS = sorted(TASK_TYPE_REF)

# ---- 팀 프로필 읽어서 대조 대상 고르기 ----
team = {}
for name in DATASETS:
    t = pd.read_csv(TEAM_DIR / name / "molecule_profile.csv")
    t["dataset"] = name
    team[name] = t

target = []
for name, t in team.items():
    sel = t if (MODE == "full" or len(t) <= SAMPLE_PER_SET) else t.sample(SAMPLE_PER_SET, random_state=SEED)
    target.append(sel)
target = pd.concat(target, ignore_index=True)
todo_smiles = sorted(set(target.smiles))

print(f"MODE = {MODE}")
print(f"대조 대상 행 : {len(target):,}")
print(f"고유 SMILES  : {len(todo_smiles):,}   <- 실제 계산은 이만큼만")

# ---- 캐시 ----
CACHE = CACHE_DIR / "verify_cache.csv"
if CACHE.exists():
    cache_df = pd.read_csv(CACHE)
    print(f"기존 캐시 {len(cache_df):,}건 재사용")
else:
    cache_df = pd.DataFrame(columns=["smiles", "our_parent", "our_scaffold",
                                     "our_n_tautomers", "our_n_protomers"])
todo = [s for s in todo_smiles if s not in set(cache_df.smiles)]
print(f"새로 계산할 분자: {len(todo):,}")


def _one(smi):
    p = to_parent(smi)
    parent = p["smiles_parent"]
    scaf = get_scaffold(parent) if parent else None
    n_t, _ = count_tautomers(parent) if parent else (np.nan, False)
    n_p = count_protomers(parent) if parent else np.nan
    return (smi, parent, scaf, n_t, n_p)


def parallel_map(fn, items, n_jobs, desc):
    if n_jobs and n_jobs > 1 and len(items) > 200:
        try:
            from joblib import Parallel, delayed
            return Parallel(n_jobs=n_jobs, batch_size=32)(
                delayed(fn)(x) for x in tqdm(items, desc=f"{desc} (병렬 {n_jobs})"))
        except Exception as e:
            print(f"  병렬 실패({type(e).__name__}) — 순차 실행으로 전환")
    return [fn(x) for x in tqdm(items, desc=desc)]


if todo:
    _TAUT_ENUM = None      # 무거운 객체가 자식 프로세스로 복사되지 않게 비운다
    _PROTONATOR = None
    res = parallel_map(_one, todo, N_JOBS, "재계산")
    cache_df = pd.concat([cache_df, pd.DataFrame(
        res, columns=["smiles", "our_parent", "our_scaffold",
                      "our_n_tautomers", "our_n_protomers"])], ignore_index=True)
    cache_df = cache_df.drop_duplicates("smiles", keep="last")
    cache_df.to_csv(CACHE, index=False)

print(f"\n캐시 총 {len(cache_df):,}건 -> {CACHE}")

MODE = verify
대조 대상 행 : 6,600
고유 SMILES  : 5,369   <- 실제 계산은 이만큼만
새로 계산할 분자: 5,369


재계산 (병렬 2):   0%|          | 0/5369 [00:00<?, ?it/s]

  병렬 실패(PicklingError) — 순차 실행으로 전환


재계산:   0%|          | 0/5369 [00:00<?, ?it/s]


캐시 총 5,369건 -> /content/drive/MyDrive/MIST/outputs/processed/_cache/verify_cache.csv


## 3. 대조

각 열의 일치율을 본다. **`B1_*_판정`(2개 이상인가) 일치율이 1차 지표**이고,
`n_*_개수` 일치율은 설정값 차이 때문에 낮게 나올 수 있으므로 참고용이다.

In [ ]:
# ============================================================================
# [셀 10] 열별 일치율 계산
# ============================================================================
m = target.merge(cache_df, on="smiles", how="left")

# 팀 열 정리 (골격 빈값은 NaN 으로 읽히므로 "" 로 맞춘다)
m["team_scaffold"] = m.scaffold.fillna("")
m["our_scaffold"] = m.our_scaffold.fillna("")
m["our_has_salt"] = m.smiles.map(has_salt_raw)
m["our_has_stereo"] = m.smiles.map(has_stereo_raw)

rows = []
for name, g in m.groupby("dataset"):
    g = g[g.our_parent.notna()]
    if not len(g):
        continue
    rows.append({
        "dataset": name, "n": len(g),
        "부모분자": round(100 * (g.our_parent == g.parent_smiles).mean(), 1),
        "골격": round(100 * (g.our_scaffold == g.team_scaffold).mean(), 1),
        "염": round(100 * (g.our_has_salt == g.has_salt.astype(bool)).mean(), 1),
        "입체": round(100 * (g.our_has_stereo == g.has_stereo.astype(bool)).mean(), 1),
        "호변_판정": round(100 * ((g.our_n_tautomers >= 2) == (g.n_tautomers >= 2)).mean(), 1),
        "양성자_판정": round(100 * ((g.our_n_protomers >= 2) == (g.n_protomers >= 2)).mean(), 1),
        "호변_개수": round(100 * (g.our_n_tautomers == g.n_tautomers).mean(), 1),
        "양성자_개수": round(100 * (g.our_n_protomers == g.n_protomers).mean(), 1),
    })

verify = pd.DataFrame(rows)
display(verify)

print("전체 가중평균 일치율 (%)")
for c in ["부모분자", "골격", "염", "입체", "호변_판정", "양성자_판정", "호변_개수", "양성자_개수"]:
    w = (verify[c] * verify.n).sum() / verify.n.sum()
    mark = "  <- 1차 지표" if c.endswith("_판정") or c in ("부모분자", "골격", "염", "입체") else ""
    print(f"  {c:12s} {w:6.1f}{mark}")

low = verify[(verify[["부모분자", "골격", "염", "입체"]] < 99).any(axis=1)]
if len(low):
    print("\n[확인 필요] 구조 관련 열의 일치율이 99% 미만인 물성:")
    display(low)
else:
    print("\n구조 관련 열(부모분자·골격·염·입체)은 전 물성 99% 이상 일치")

,dataset,n,부모분자,골격,염,입체,호변_판정,양성자_판정,호변_개수,양성자_개수
0,ames,300,99.7,100.0,100.0,100.0,99.7,100.0,96.7,97.3
1,bbb_martins,300,99.7,99.7,100.0,100.0,100.0,100.0,91.7,85.0
2,bioavailability_ma,300,98.3,98.7,100.0,100.0,100.0,100.0,96.3,83.0
3,caco2_wang,300,100.0,100.0,100.0,100.0,100.0,100.0,78.7,69.0
4,clearance_hepatocyte_az,300,100.0,100.0,100.0,100.0,100.0,100.0,94.7,71.0
5,clearance_microsome_az,300,100.0,100.0,100.0,100.0,100.0,100.0,96.0,67.7
6,cyp2c9_substrate_carbonmangels,300,99.7,100.0,100.0,100.0,100.0,100.0,94.7,89.0
7,cyp2c9_veith,300,98.3,98.3,100.0,100.0,99.3,100.0,95.7,84.3
8,cyp2d6_substrate_carbonmangels,300,100.0,100.0,100.0,100.0,100.0,100.0,96.7,90.3
9,cyp2d6_veith,300,99.7,99.7,100.0,100.0,100.0,100.0,95.7,83.3


전체 가중평균 일치율 (%)
  부모분자           99.5  <- 1차 지표
  골격             99.7  <- 1차 지표
  염             100.0  <- 1차 지표
  입체            100.0  <- 1차 지표
  호변_판정          99.9  <- 1차 지표
  양성자_판정        100.0  <- 1차 지표
  호변_개수          94.0
  양성자_개수         82.7

[확인 필요] 구조 관련 열의 일치율이 99% 미만인 물성:


,dataset,n,부모분자,골격,염,입체,호변_판정,양성자_판정,호변_개수,양성자_개수
2,bioavailability_ma,300,98.3,98.7,100.0,100.0,100.0,100.0,96.3,83.0
7,cyp2c9_veith,300,98.3,98.3,100.0,100.0,99.3,100.0,95.7,84.3
15,hia_hou,300,98.7,99.0,100.0,100.0,100.0,100.0,97.3,90.7
20,solubility_aqsoldb,300,98.0,100.0,100.0,100.0,99.7,99.7,96.7,96.7


In [ ]:
# ============================================================================
# [셀 11] 불일치 사례 직접 보기 — 왜 갈렸는지 눈으로 확인
# ============================================================================
def show_mismatch(col_ours, col_theirs, label, k=5):
    bad = m[m.our_parent.notna() & (m[col_ours] != m[col_theirs])]
    print(f"\n### {label}: 불일치 {len(bad):,} / {int(m.our_parent.notna().sum()):,}")
    for _, r in bad.head(k).iterrows():
        print(f"  [{r.dataset}] {str(r.smiles)[:60]}")
        print(f"      팀원: {r[col_theirs]}")
        print(f"      우리: {r[col_ours]}")

show_mismatch("our_parent", "parent_smiles", "부모 분자")
show_mismatch("our_scaffold", "team_scaffold", "골격")

# 개수 차이는 분포로 본다 (설정값 차이라면 한쪽으로 치우친 모양이 나온다)
d_t = (m.our_n_tautomers - m.n_tautomers).dropna()
d_p = (m.our_n_protomers - m.n_protomers).dropna()
print(f"\n### 개수 차이 (우리 - 팀원)")
print(f"  호변이성질체 : 중앙값 {d_t.median():+.0f}, 같음 {100*(d_t==0).mean():.1f}%, "
      f"우리가 많음 {100*(d_t>0).mean():.1f}%, 적음 {100*(d_t<0).mean():.1f}%")
print(f"  양성자화     : 중앙값 {d_p.median():+.0f}, 같음 {100*(d_p==0).mean():.1f}%, "
      f"우리가 많음 {100*(d_p>0).mean():.1f}%, 적음 {100*(d_p<0).mean():.1f}%")
print("\n한쪽으로 치우쳐 있으면 열거 상한(maxTautomers)이나 pH 창 설정이 다른 것이고,")
print("양방향으로 흩어져 있으면 구현 자체가 다른 것이다.")


### 부모 분자: 불일치 35 / 6,600
  [ames] CS(=O)c1ccc(Cl)cc1
      팀원: CS(=O)c1ccc(Cl)cc1
      우리: C[S+]([O-])c1ccc(Cl)cc1
  [bbb_martins] COc1ccc2nc(S(=O)Cc3ncc(C)c(OC)c3C)[nH]c2c1
      팀원: COc1ccc2nc(S(=O)Cc3ncc(C)c(OC)c3C)[nH]c2c1
      우리: COc1ccc2nc([S+]([O-])Cc3ncc(C)c(OC)c3C)[nH]c2c1
  [bioavailability_ma] O=C1C(CCS(=O)c2ccccc2)C(=O)N(c2ccccc2)N1c1ccccc1
      팀원: O=C1C(CCS(=O)c2ccccc2)C(=O)N(c2ccccc2)N1c1ccccc1
      우리: O=C1C(CC[S+]([O-])c2ccccc2)C(=O)N(c2ccccc2)N1c1ccccc1
  [bioavailability_ma] CC1=C(CC(=O)O)c2cc(F)ccc2/C1=C\c1ccc(S(C)=O)cc1
      팀원: CC1=C(CC(=O)O)c2cc(F)ccc2/C1=C\c1ccc(S(C)=O)cc1
      우리: CC1=C(CC(=O)O)c2cc(F)ccc2/C1=C\c1ccc([S+](C)[O-])cc1
  [bioavailability_ma] COCCCOc1ccnc(CS(=O)c2nc3ccccc3[nH]2)c1C
      팀원: COCCCOc1ccnc(CS(=O)c2nc3ccccc3[nH]2)c1C
      우리: COCCCOc1ccnc(C[S+]([O-])c2nc3ccccc3[nH]2)c1C

### 골격: 불일치 19 / 6,600
  [bbb_martins] COc1ccc2nc(S(=O)Cc3ncc(C)c(OC)c3C)[nH]c2c1
      팀원: O=S(Cc1ccccn1)c1nc2ccccc2[nH]1
      우리: c1ccc(C[SH+]c2nc3ccccc3

## 4. 게이트 수치 대조

계획서 5.4절의 **10% 게이트**는 축 선택의 근거다. 팀원이 낸 비율과 우리 표본 비율을 비교한다.
`verify` 모드에서는 표본 오차가 있으므로 (300개 기준 95% 신뢰구간 약 ±3%p 이내)
**5%p 이상 차이 나는 칸만** 확인 대상으로 본다.

In [ ]:
# ============================================================================
# [셀 12] 팀 게이트 수치 vs 우리 재계산 비율
# ============================================================================
pre = pd.read_csv(TEAM_DIR / "prescreen_summary.csv")
team_pct = pre.set_index("dataset")[["pct_multi_tautomer", "pct_multi_protomer",
                                     "pct_with_salt", "pct_with_stereo"]]

ours = m[m.our_parent.notna()].groupby("dataset").agg(
    our_tautomer=("our_n_tautomers", lambda s: 100 * (s >= 2).mean()),
    our_protomer=("our_n_protomers", lambda s: 100 * (s >= 2).mean()),
    our_salt=("our_has_salt", lambda s: 100 * s.mean()),
    our_stereo=("our_has_stereo", lambda s: 100 * s.mean()),
).round(1)

cmp = team_pct.join(ours)
for a, t, o in [("호변", "pct_multi_tautomer", "our_tautomer"),
                ("양성자", "pct_multi_protomer", "our_protomer"),
                ("염", "pct_with_salt", "our_salt"),
                ("입체", "pct_with_stereo", "our_stereo")]:
    cmp[f"차이_{a}"] = (cmp[o] - cmp[t]).round(1)
    # 10% 게이트 판정이 뒤집히는가 — 이게 진짜 중요한 것
    cmp[f"판정뒤집힘_{a}"] = (cmp[t] >= 10) != (cmp[o] >= 10)

display(cmp)

flip = cmp[[c for c in cmp.columns if c.startswith("판정뒤집힘")]].any(axis=1)
if flip.any():
    print("\n[중요] 10% 게이트 판정이 뒤집히는 물성 — 축 선택이 달라진다:")
    display(cmp[flip])
    print("MODE='full' 로 전수 재계산해서 어느 쪽이 맞는지 확정하세요.")
else:
    print("\n✅ 게이트 판정(10% 기준)이 뒤집히는 물성 없음 — 팀 수치를 그대로 써도 된다.")

big = cmp[[c for c in cmp.columns if c.startswith("차이_")]].abs().max(axis=1) >= 5
if big.any():
    print(f"\n[참고] 5%p 이상 차이 나는 물성 {int(big.sum())}종:", list(cmp[big].index))

verify.to_csv(REPORT_DIR / "09_verification_columns.csv", index=False)
cmp.reset_index().to_csv(REPORT_DIR / "09_verification_gate.csv", index=False)
print(f"\n저장: reports/09_verification_columns.csv, reports/09_verification_gate.csv")
if MODE != "full":
    print(f"(표본 {SAMPLE_PER_SET}개/물성 기준. 논문에 실을 수치는 노트북 02가 쓰는 팀 전수 값이다.)")

,pct_multi_tautomer,pct_multi_protomer,pct_with_salt,pct_with_stereo,our_tautomer,our_protomer,our_salt,our_stereo,차이_호변,판정뒤집힘_호변,차이_양성자,판정뒤집힘_양성자,차이_염,판정뒤집힘_염,차이_입체,판정뒤집힘_입체
dataset,,,,,,,,,,,,,,,,
ames,44.44,48.04,0.00,14.04,46.7,48.7,0.0,14.0,2.3,False,0.7,False,0.0,False,-0.0,False
bbb_martins,65.52,82.86,5.17,35.32,62.7,82.7,4.7,34.7,-2.8,False,-0.2,False,-0.5,False,-0.6,False
bioavailability_ma,67.34,86.41,0.16,40.94,69.3,87.0,0.0,38.0,2.0,False,0.6,False,-0.2,False,-2.9,False
caco2_wang,82.09,88.02,0.99,54.07,82.7,85.3,1.3,58.3,0.6,False,-2.7,False,0.3,False,4.2,False
clearance_hepatocyte_az,82.61,92.00,0.00,38.58,80.0,92.3,0.0,35.0,-2.6,False,0.3,False,0.0,False,-3.6,False
clearance_microsome_az,86.39,90.02,0.00,34.21,89.0,91.3,0.0,34.0,2.6,False,1.3,False,0.0,False,-0.2,False
cyp2c9_substrate_carbonmangels,63.53,82.96,0.00,59.49,61.3,82.3,0.0,56.7,-2.2,False,-0.7,False,0.0,False,-2.8,False
cyp2c9_veith,70.46,89.55,4.45,30.51,68.3,91.0,5.0,26.0,-2.2,False,1.5,False,0.5,False,-4.5,False
cyp2d6_substrate_carbonmangels,63.57,83.06,0.00,59.37,61.0,79.3,0.0,61.3,-2.6,False,-3.8,False,0.0,False,1.9,False



✅ 게이트 판정(10% 기준)이 뒤집히는 물성 없음 — 팀 수치를 그대로 써도 된다.

[참고] 5%p 이상 차이 나는 물성 1종: ['cyp2d6_veith']

저장: reports/09_verification_columns.csv, reports/09_verification_gate.csv
(표본 300개/물성 기준. 논문에 실을 수치는 노트북 02가 쓰는 팀 전수 값이다.)


---

## ✅ 검증 완료

- `reports/09_verification_columns.csv` — 열별 일치율
- `reports/09_verification_gate.csv` — 게이트 비율 대조 (10% 판정 뒤집힘 여부 포함)

**판단 기준**

- 구조 열(부모분자·골격·염·입체) 일치율이 99% 이상이고 게이트 판정이 안 뒤집히면
  → 팀 산출물을 그대로 쓴다. 노트북 02로 진행.
- 게이트 판정이 뒤집히는 물성이 있으면 → `MODE = "full"` 로 전수 재계산 후 어느 쪽이 맞는지 확정.

**다음**: `02_dedup_allowance_split.ipynb` (태스크 3·7·8)

In [ ]:
# 등록형 포함 수치 검증
import pandas as pd, numpy as np
from pathlib import Path
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm
from google.colab import drive
drive.mount("/content/drive")
RDLogger.DisableLog("rdApp.*")

TEAM = Path("/content/drive/MyDrive/MIST/data/processed")

# ── dimorphite-dl 어댑터 (01 노트북과 동일 설정) ──────────────────────────
PH_MIN, PH_MAX, PKA_PRECISION, MAX_PROTOMERS = 6.4, 8.4, 1.0, 16
_PROTONATOR = None

def _init_protonator():
    global _PROTONATOR
    try:                                   # 2.x: 모듈 함수
        from dimorphite_dl import protonate_smiles as _ps
        def _call(s):
            try:
                return list(_ps(s, ph_min=PH_MIN, ph_max=PH_MAX,
                                precision=PKA_PRECISION, max_variants=MAX_PROTOMERS))
            except TypeError:
                return list(_ps(s))
        _PROTONATOR = _call
        return
    except Exception:
        pass
    from dimorphite_dl import DimorphiteDL   # 1.x: 클래스
    try:
        _dl = DimorphiteDL(min_ph=PH_MIN, max_ph=PH_MAX, max_variants=MAX_PROTOMERS,
                           pka_precision=PKA_PRECISION, label_states=False, silent=True)
    except TypeError:
        _dl = DimorphiteDL(min_ph=PH_MIN, max_ph=PH_MAX, pka_precision=PKA_PRECISION)
    _PROTONATOR = lambda s: list(_dl.protonate(s))

def protomer_states(smiles):
    """지정 pH 범위에서 가능한 양성자화 상태들의 '정준 SMILES 집합'."""
    global _PROTONATOR
    if _PROTONATOR is None:
        _init_protonator()
    if not isinstance(smiles, str):
        return set()
    try:
        out = _PROTONATOR(smiles)
    except Exception:
        return set()
    canon = set()
    for s in out:
        m = Chem.MolFromSmiles(s)
        if m is not None:
            canon.add(Chem.MolToSmiles(m))
    return canon

_init_protonator()
print("동작 확인:", sorted(protomer_states("CC(=O)O")), "\n")

# ── 애매한 부분집합만 재계산 ──────────────────────────────────────────────
# 팀원의 n_protomers 가 2 이상이면 이미 '변형 가능'이라 볼 필요가 없다.
# n_protomers <= 1 인 분자만 골라, 그 단일 상태가 등록형과 다른지 확인한다.
N = 200      # 물성당 표본 크기

rows = []
for prof_path in tqdm(sorted(TEAM.glob("*/molecule_profile.csv")), desc="물성"):
    d = prof_path.parent.name
    t = pd.read_csv(prof_path)
    base = 100 * (t.n_protomers >= 2).mean()          # 팀원 정의 (열거만)
    one = t[t.n_protomers <= 1]                       # 애매한 부분집합
    share = len(one) / len(t)                         # 전체에서 차지하는 비율

    if len(one) == 0:
        rows.append({"dataset": d, "열거만": round(base, 1), "단일상태_비율": 0.0,
                     "그중_원본과다름": np.nan, "등록형포함_추정": round(base, 1)})
        continue

    uniq = one.drop_duplicates("parent_smiles")
    s = uniq.sample(min(N, len(uniq)), random_state=42)

    diff = 0
    for p in s.parent_smiles:
        st = protomer_states(p)
        canon = Chem.MolToSmiles(Chem.MolFromSmiles(p)) if isinstance(p, str) else None
        if st and canon not in st:       # 단일 상태인데 등록형과 다르다 = 비교할 상태가 2개
            diff += 1
    frac = diff / len(s)

    rows.append({"dataset": d, "열거만": round(base, 1),
                 "단일상태_비율": round(100 * share, 1),
                 "그중_원본과다름": round(100 * frac, 1),
                 "등록형포함_추정": round(base + frac * 100 * share, 1)})

r = pd.DataFrame(rows)
print(r.to_string(index=False))
print(f"\n중앙값: 열거만 {r.열거만.median():.1f}%  ->  등록형포함 {r.등록형포함_추정.median():.1f}%")
print(f"10% 게이트 통과: 열거만 {int((r.열거만 >= 10).sum())}/22  ->  "
      f"등록형포함 {int((r.등록형포함_추정 >= 10).sum())}/22")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
동작 확인: ['CC(=O)[O-]'] 



물성:   0%|          | 0/22 [00:00<?, ?it/s]

                       dataset  열거만  단일상태_비율  그중_원본과다름  등록형포함_추정
                          ames 48.0     52.0      13.5      55.1
                   bbb_martins 82.9     17.1      11.0      84.7
            bioavailability_ma 86.4     13.6      34.5      91.1
                    caco2_wang 88.0     12.0      29.4      91.5
       clearance_hepatocyte_az 92.0      8.0      49.4      95.9
        clearance_microsome_az 90.0     10.0      73.6      97.4
cyp2c9_substrate_carbonmangels 83.0     17.0      23.7      87.0
                  cyp2c9_veith 89.5     10.5      39.5      93.7
cyp2d6_substrate_carbonmangels 83.1     16.9      23.9      87.1
                  cyp2d6_veith 89.1     10.9      37.5      93.2
cyp3a4_substrate_carbonmangels 83.3     16.7      24.1      87.3
                  cyp3a4_veith 89.6     10.4      41.5      93.9
                          dili 82.9     17.1      33.3      88.6
               half_life_obach 90.1      9.9      40.0      94.1
                         

In [ ]:
# cyp2d6_veith 5%p 차이는 어느 축인지 확인
print(cmp.loc["cyp2d6_veith"].to_string())

pct_multi_tautomer    71.46
pct_multi_protomer     89.1
pct_with_salt          4.18
pct_with_stereo       29.11
our_tautomer           63.0
our_protomer           87.0
our_salt                4.3
our_stereo             22.0
차이_호변                  -8.5
판정뒤집힘_호변              False
차이_양성자                 -2.1
판정뒤집힘_양성자             False
차이_염                    0.1
판정뒤집힘_염               False
차이_입체                  -7.1
판정뒤집힘_입체              False


In [ ]:
# 01 노트북 셀 10을 돌린 커널에서 (m 변수 재사용)
sub = m[(m.dataset == "cyp2d6_veith") & (m.our_has_stereo != m.has_stereo.astype(bool))]
print(f"입체 판정 불일치 {len(sub)}건\n")
for _, r in sub.head(12).iterrows():
    chars = [c for c in ("@", "/", "\\") if c in r.smiles]
    print(f"  팀={str(r.has_stereo):5s} 우리={str(r.our_has_stereo):5s} 기호={chars}  {r.smiles[:75]}")

입체 판정 불일치 0건

